# Monitoring, Incident Response, and Runbooks
Companion notebook for the **Monitoring, Incident Response, and Runbooks** section of Chapter 12.

The operating rule: **if a signal moves, someone should know what to do next**. This notebook loads the real `incident-runbook.yaml` shipped alongside it -- the book's example, unchanged.

In [1]:
print(open("incident-runbook.yaml").read())

incident: model_timeout
detect:
  - p95_latency_ms > 15000 for 10m
  - timeout_rate > 5%
mitigate:
  - switch_model: "fallback-small"
  - reduce_max_tokens: 4000
  - disable_optional_tools: true
rollback:
  - revert_release: previous_stable
escalate:
  - oncall_ml
  - platform_oncall



In [2]:
import yaml

runbook = yaml.safe_load(open("incident-runbook.yaml"))
print(runbook)

{'incident': 'model_timeout', 'detect': ['p95_latency_ms > 15000 for 10m', 'timeout_rate > 5%'], 'mitigate': [{'switch_model': 'fallback-small'}, {'reduce_max_tokens': 4000}, {'disable_optional_tools': True}], 'rollback': [{'revert_release': 'previous_stable'}], 'escalate': ['oncall_ml', 'platform_oncall']}


A dashboard's four groups -- **latency, errors, saturation, business outcomes** -- as one metrics snapshot, plus a detector that checks it against the runbook's `detect` thresholds:

In [3]:
def parse_detect_rule(rule: str):
    # book's rules look like "p95_latency_ms > 15000 for 10m" or "timeout_rate > 5%"
    field, _, rest = rule.partition(">")
    field = field.strip()
    threshold_part = rest.split("for")[0].strip().rstrip("%")
    return field, float(threshold_part)


def detect_incident(runbook: dict, metrics: dict) -> bool:
    return any(
        metrics.get(parse_detect_rule(rule)[0], 0) > parse_detect_rule(rule)[1]
        for rule in runbook["detect"]
    )


healthy_metrics = {"p95_latency_ms": 4200, "timeout_rate": 1.2}
incident_metrics = {"p95_latency_ms": 19000, "timeout_rate": 6.5}

print("healthy window triggers incident:", detect_incident(runbook, healthy_metrics))
print("bad window triggers incident:     ", detect_incident(runbook, incident_metrics))

healthy window triggers incident: False
bad window triggers incident:      True


A runbook executor -- **acknowledge, contain, communicate, recover, review** -- that walks the same `mitigate` / `rollback` / `escalate` steps the YAML already lists, so the response is the config, not a separate document nobody keeps in sync:

In [4]:
def run_incident_response(runbook: dict, metrics: dict) -> dict:
    if not detect_incident(runbook, metrics):
        return {"status": "no_incident"}

    actions_taken = []
    for step in runbook["mitigate"]:
        for action, value in step.items():
            actions_taken.append(f"{action}={value}")

    return {
        "status": "mitigated",
        "incident": runbook["incident"],
        "actions_taken": actions_taken,
        "escalated_to": runbook["escalate"],
    }


print(run_incident_response(runbook, incident_metrics))

{'status': 'mitigated', 'incident': 'model_timeout', 'actions_taken': ['switch_model=fallback-small', 'reduce_max_tokens=4000', 'disable_optional_tools=True'], 'escalated_to': ['oncall_ml', 'platform_oncall']}


A short runbook that's actually followed beats a perfect document nobody opens -- confirm the incident, mitigate, roll back if needed, review logs and traces, escalate on the criteria decided *before* the incident, and always write a short post-incident note: trigger, impact, timeline, root cause, mitigation, permanent fix.